In [ ]:
# @title 1. Setup & Install
!pip install ultralytics -q

import torch
import cv2
import numpy as np
import collections
import os
from google.colab.patches import cv2_imshow
from google.colab import output
from ultralytics import YOLO

# Check for GPU
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    DEVICE = 0
else:
    print("⚠️ WARNING: No GPU detected. It will be slower.")
    DEVICE = 'cpu'

In [ ]:
# @title 2. Upload Photo
from google.colab import files

uploaded = files.upload()
IMAGE_FILENAME = list(uploaded.keys())[0]
print(f"✅ Ready to process: {IMAGE_FILENAME}")

In [ ]:
# @title 3. Draw Zone
import base64 # Import the base64 module
from IPython.display import HTML, display # Import HTML

img = cv2.imread(IMAGE_FILENAME)
resize_scale = 0.5 # Resize for easier drawing on screen
h, w = img.shape[:2]
img_resized = cv2.resize(img, (int(w * resize_scale), int(h * resize_scale)))
_, buffer = cv2.imencode('.jpg', img_resized)
img_str = base64.b64encode(buffer).decode()

js_code = f"""
<canvas id="canvas" width="{img_resized.shape[1]}" height="{img_resized.shape[0]}" style="border:2px solid red; cursor:crosshair;"></canvas>
<br><button onclick="finish()" style="font-size:16px; margin:5px;">✅ Finish Zone</button>
<p id="msg" style="color:green;"></p>
<script>
  var canvas = document.getElementById('canvas');
  var ctx = canvas.getContext('2d');
  var img = new Image();
  img.src = "data:image/jpeg;base64,{img_str}";
  var points = [];
  img.onload = function() {{ ctx.drawImage(img, 0, 0); }};
  canvas.onclick = function(e) {{
    var rect = canvas.getBoundingClientRect();
    var x = e.clientX - rect.left;
    var y = e.clientY - rect.top;
    points.push([x, y]);
    ctx.fillStyle = "lime"; ctx.beginPath(); ctx.arc(x, y, 4, 0, 2 * Math.PI); ctx.fill();
    if (points.length > 1) {{
      ctx.beginPath(); ctx.moveTo(points[points.length-2][0], points[points.length-2][1]);
      ctx.lineTo(x, y); ctx.strokeStyle = "lime"; ctx.lineWidth = 2; ctx.stroke();
    }}
  }};
  function finish() {{
    if (points.length > 2) {{
      ctx.beginPath(); ctx.moveTo(points[points.length-1][0], points[points.length-1][1]);
      ctx.lineTo(points[0][0], points[0][1]); ctx.stroke();
      google.colab.kernel.invokeFunction('notebook.save_points', [points], {{}});
      document.getElementById('msg').innerText = "Zone Saved!";
    }}
  }}
</script>
"""

zone_polygon = []
def save_points(pts):
    global zone_polygon, resize_scale
    scale = 1 / resize_scale
    zone_polygon = [[int(p[0] * scale), int(p[1] * scale)] for p in pts]
    print(f"✅ Zone defined with {len(zone_polygon)} points.")

output.register_callback('notebook.save_points', save_points)
display(HTML(js_code))

In [ ]:

CONFIDENCE = 0.15   # Low confidence to catch hidden items
IMGSZ = 1280        # High Resolution for small objects

print("🚀 Loading YOLOv8-ExtraLarge...")
model = YOLO('yolov8x.pt')

# Load Image
img = cv2.imread(IMAGE_FILENAME)
h, w = img.shape[:2]

# 1. INFERENCE
print("▶️ Scanning full image...")
results = model.predict(img, conf=CONFIDENCE, imgsz=IMGSZ, device=DEVICE, verbose=False)

if results[0].boxes.id is not None or len(results[0].boxes) > 0:
    boxes = results[0].boxes.xyxy.cpu().numpy()
    clss = results[0].boxes.cls.int().cpu().tolist()
    names = results[0].names

    # Organize items by their Name (e.g., 'bottle': [box1, box2])
    grouped_products = collections.defaultdict(list)

    for box, cls in zip(boxes, clss):
            name = names[cls]
            grouped_products[name].append(box)

    # 2. GROUPING LOGIC (The "Mega Box")
    total_groups = 0

    for prod_name, prod_boxes in grouped_products.items():
        count = len(prod_boxes)
        if count > 0:
            total_groups += 1

            # Find the boundaries that cover ALL items of this type
            min_x = min([b[0] for b in prod_boxes])
            min_y = min([b[1] for b in prod_boxes])
            max_x = max([b[2] for b in prod_boxes])
            max_y = max([b[3] for b in prod_boxes])

            # Draw the Big Group Box
            cv2.rectangle(img, (int(min_x), int(min_y)), (int(max_x), int(max_y)), (0, 165, 255), 3)

            # Label: "BOTTLES: 5 items"
            label = f"{prod_name.upper()}S: {count} items"

            # Draw label background for readability
            (w_text, h_text), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            cv2.rectangle(img, (int(min_x), int(min_y)-30), (int(min_x)+w_text, int(min_y)), (0, 165, 255), -1)

            cv2.putText(img, label, (int(min_x), int(min_y)-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    print(f"✅ Found {total_groups} groups of products.")
else:
    print("❌ No target products found.")

# Display Result
resize_preview = 0.5
h, w = img.shape[:2]
preview_img = cv2.resize(img, (int(w * resize_preview), int(h * resize_preview)))
cv2_imshow(preview_img)

# Save
output_filename = "grouped_" + IMAGE_FILENAME
cv2.imwrite(output_filename, img)
print(f"💾 Saved as {output_filename}")